# ハイパーパラメータの探索
モデルの設定(ハイパーパラメータ)の組み合わせを、交差検証で自動的に比べて選ぶ方法を学びます。`GridSearchCV` で候補をまとめて評価し、最も良い組み合わせとその精度を確かめます。

## 学習目標
- ハイパーパラメータと、学習で決まるパラメータの違いを説明できる
- `GridSearchCV` で、ハイパーパラメータの組み合わせを交差検証で比べられる
- 探索の結果(`best_params_`・`best_score_`・`cv_results_`)を読み取れる
- 探索に予測用のデータを使ってはいけない理由を説明できる

## ハイパーパラメータとは

| 種類 | 例 | 決め方 |
|---|---|---|
| **パラメータ** | ロジスティック回帰の係数、決定木の分岐の質問 | 学習(`fit`)によってデータから自動で決まる |
| **ハイパーパラメータ** | 決定木の深さ `max_depth`、葉の最小人数 `min_samples_leaf` | 学習の前に人が決める |

前の講座では、深さを 1 つずつ変えて交差検証で比べました。複数のハイパーパラメータの組み合わせを試すには、`GridSearchCV` を使うと便利です。指定した候補の **すべての組み合わせ**(グリッド)を交差検証で評価し、最も良いものを選んでくれます。

## GridSearchCV で探索する

ランダムフォレストの `max_depth`(3・5・7)と `min_samples_leaf`(1・3・5)の 3 × 3 = 9 通りを探索します。各組み合わせで 5 分割交差検証を行うので、学習は 45 回行われます(実行に 10 秒ほどかかります)。

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# 前の講座で作った特徴量の表を読み込み、特徴量 X と正解 y に分ける
df = pd.read_csv("train_prepared.csv")
X = df.drop(columns=["PassengerId", "Survived"])
y = df["Survived"]

# 5 分割の分け方(前の講座と同じ)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# 探索する候補(ハイパーパラメータの名前 → 試す値のリスト)
param_grid = {
    "max_depth": [3, 5, 7],
    "min_samples_leaf": [1, 3, 5],
}

# すべての組み合わせを交差検証で評価する
search = GridSearchCV(RandomForestClassifier(n_estimators=100, random_state=0), param_grid, cv=cv)
search.fit(X, y)

# 最も良い組み合わせと、そのときの交差検証の正解率(平均)
print("最も良い組み合わせ:", search.best_params_)
print("交差検証の正解率  :", round(search.best_score_, 3))

最も良い組み合わせ: {'max_depth': 5, 'min_samples_leaf': 5}
交差検証の正解率  : 0.818


`max_depth=5`・`min_samples_leaf=5` の組み合わせが最も良く、交差検証の正解率は 81.8% です。

すべての組み合わせの結果は `cv_results_` に入っています。表にして確かめます。

In [2]:
# cv_results_ を表にして、組み合わせと交差検証の正解率(平均)の列だけを表示する
results = pd.DataFrame(search.cv_results_)
results[["param_max_depth", "param_min_samples_leaf", "mean_test_score"]].round(3)

   param_max_depth  param_min_samples_leaf  mean_test_score
0                3                       1            0.809
1                3                       3            0.809
2                3                       5            0.809
3                5                       1            0.817
4                5                       3            0.818
5                5                       5            0.818
6                7                       1            0.805
7                7                       3            0.813
8                7                       5            0.817


- 深さ 3 はどの組み合わせでも 80.9% で、やや単純すぎます
- 深さ 5 は 81.7〜81.8% と安定して高い値です
- 深さ 7 は `min_samples_leaf=1`(80.5%)では下がり、葉の人数を増やすと(81.7%)回復します。深い木でも、葉の人数を制限すると過学習が抑えられます

最も良い組み合わせと前の講座の設定(深さ 5 だけ指定:81.7%)との差は 0.1% で、ほとんど変わりません。タイタニックのデータでは、ハイパーパラメータを変えても精度は 81〜82% 前後で頭打ちになります。**差が交差検証の標準偏差(約 2%)より十分小さいときは、偶然の差かもしれない** ことに注意しましょう。

`search.best_estimator_` には、最も良い組み合わせですべての学習用データを使って学習し直したモデルが入っています。このモデルをそのまま予測に使えます(単元5)。

## 演習

決定木(`random_state=0`)の `max_depth`(2・3・4・5・6)と `min_samples_leaf`(1・5・10・20)のすべての組み合わせを、解説と同じ分け方の `GridSearchCV` で探索してください。最も良い組み合わせと、そのときの交差検証の正解率(小数第3位まで)を、この順に表示してください。

In [ ]:
# @starter
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

# 特徴量の表を読み込み、特徴量 X と正解 y に分ける
df = pd.read_csv("train_prepared.csv")
X = df.drop(columns=["PassengerId", "Survived"])
y = df["Survived"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# 探索する候補(____ に、ハイパーパラメータの名前と試す値のリストを書く)
param_grid = {
    ____,
    ____,
}

# 決定木で、すべての組み合わせを交差検証で評価する
search = GridSearchCV(DecisionTreeClassifier(random_state=0), param_grid, cv=cv)
search.fit(X, y)

# 最も良い組み合わせと、交差検証の正解率を表示する
print(search.best_params_)
print(round(search.best_score_, 3))

In [3]:
# @solution
# @title: 決定木のハイパーパラメータの探索
# @hint: param_grid は {"max_depth": [2, 3, 4, 5, 6], "min_samples_leaf": [1, 5, 10, 20]} のような辞書です。
# @explanation: 5 × 4 = 20 通りの中で、max_depth=5・min_samples_leaf=5 の決定木が最も良く、交差検証の正解率は 81.8% です。1本の決定木でも、ハイパーパラメータを調整するとランダムフォレストと同じくらいの精度になりました。
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

# 特徴量の表を読み込み、特徴量 X と正解 y に分ける
df = pd.read_csv("train_prepared.csv")
X = df.drop(columns=["PassengerId", "Survived"])
y = df["Survived"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# 探索する候補(ハイパーパラメータの名前 → 試す値のリスト)
param_grid = {
    "max_depth": [2, 3, 4, 5, 6],
    "min_samples_leaf": [1, 5, 10, 20],
}

# 決定木で、すべての組み合わせを交差検証で評価する
search = GridSearchCV(DecisionTreeClassifier(random_state=0), param_grid, cv=cv)
search.fit(X, y)

# 最も良い組み合わせと、交差検証の正解率を表示する
print(search.best_params_)
print(round(search.best_score_, 3))

{'max_depth': 5, 'min_samples_leaf': 5}
0.818


前の演習と同じ決定木の探索で、`cv_results_` を表にし、交差検証の正解率(`mean_test_score`)が **高い順に上位 3 つ** の組み合わせを表示してください(列は `param_max_depth`・`param_min_samples_leaf`・`mean_test_score`、正解率は小数第3位まで)。

In [ ]:
# @starter
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

# 特徴量の表を読み込み、決定木のハイパーパラメータを探索する
df = pd.read_csv("train_prepared.csv")
X = df.drop(columns=["PassengerId", "Survived"])
y = df["Survived"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
param_grid = {"max_depth": [2, 3, 4, 5, 6], "min_samples_leaf": [1, 5, 10, 20]}
search = GridSearchCV(DecisionTreeClassifier(random_state=0), param_grid, cv=cv).fit(X, y)

# 結果を表にして、必要な列だけを取り出す(____ に、結果が入っている属性の名前を入れる)
results = pd.DataFrame(search.____)
results = results[["param_max_depth", "param_min_samples_leaf", "mean_test_score"]]

# 正解率の高い順に並べて、上位 3 つを表示する(____ に、並べ替えの式を書く)
print(____.head(3).round(3))

In [4]:
# @solution
# @title: 探索の結果の上位 3 つ
# @hint: 探索の結果は cv_results_ に入っています。results.sort_values("mean_test_score", ascending=False) で正解率の高い順に並びます。
# @explanation: 上位の組み合わせの正解率の差は 1% 未満です。最も良い組み合わせだけでなく、上位の組み合わせの傾向(深さ 5〜6 で、葉の人数を 5〜10 人に制限する)を見ることで、安定して良い設定がわかります。
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

# 特徴量の表を読み込み、決定木のハイパーパラメータを探索する
df = pd.read_csv("train_prepared.csv")
X = df.drop(columns=["PassengerId", "Survived"])
y = df["Survived"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
param_grid = {"max_depth": [2, 3, 4, 5, 6], "min_samples_leaf": [1, 5, 10, 20]}
search = GridSearchCV(DecisionTreeClassifier(random_state=0), param_grid, cv=cv).fit(X, y)

# cv_results_ を表にして、必要な列だけを取り出す
results = pd.DataFrame(search.cv_results_)
results = results[["param_max_depth", "param_min_samples_leaf", "mean_test_score"]]

# 正解率の高い順に並べて、上位 3 つを表示する
print(results.sort_values("mean_test_score", ascending=False).head(3).round(3))

    param_max_depth  param_min_samples_leaf  mean_test_score
13                5                       5            0.818
14                5                      10            0.817
17                6                       5            0.817


<!-- @question
type: choice
answer: 予測用のデータは本番で予測するデータで、その結果を見て設定を選ぶと、本番の精度を正しく見積もれなくなるため。
title: 探索に使うデータ
hint: 予測用のデータは「まだ見たことのない、本番で予測するデータ」の役割です。それを使って設定を選ぶと、どうなるでしょうか。
explanation: ハイパーパラメータは、学習用のデータの中での交差検証で選びます。予測用(本番)のデータの結果を見て設定を選ぶと、その設定はたまたま予測用のデータに合ったものになり、新しいデータでの本当の精度より高く見積もってしまいます(これもリークの一種です)。そもそも本番では、予測するデータの正解は手元にありません。
-->
ハイパーパラメータの探索で、予測用のデータ(本番で予測するデータ)の正解率を見て設定を選んではいけない理由として、最も適切なものを選べ。

- 予測用のデータは本番で予測するデータで、その結果を見て設定を選ぶと、本番の精度を正しく見積もれなくなるため。
- 予測用のデータは行数が少ないので、交差検証ができないため。
- GridSearchCV は、予測用のデータを受け取れないため。
- 予測用のデータで選ぶと、学習に時間がかかりすぎるため。

## まとめ

- ハイパーパラメータは学習の前に人が決める設定。`GridSearchCV` で候補のすべての組み合わせを交差検証で比べ、`best_params_`・`best_score_` で結果を確かめる。
- タイタニックのデータでは、調整しても交差検証の正解率は 81〜82% 前後。差が標準偏差より小さいときは偶然の差の可能性がある。
- 探索は学習用のデータの中で行う。予測用(本番)のデータを見て設定を選ばない。